## extract from csv volume

In [0]:
%restart_python

In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql.functions import current_timestamp, lit
import yaml
from dotenv import load_dotenv
import os

In [0]:
load_dotenv(
    "/Workspace/Users/kahledtrojan@gmail.com/databricks_lakehouse_pipeline_de/bike_lakehouse/config/dev/secrets/.env"
)


In [0]:
with open("/Workspace/Users/kahledtrojan@gmail.com/databricks_lakehouse_pipeline_de/bike_lakehouse/config.yaml","r") as f:
    config = yaml.safe_load(f)

In [0]:
crm_tables = ["cust_info","prd_info","sales_details"]
erp_tables = ["CUST_AZ12","LOC_A101","PX_CAT_G1V2"]





def load_to_bronze(tables,source_system):
    for table in tables:
        df = (
            spark.read.option("header","true")
            .option("inferSchema","true")
            .csv(f"/Volumes/bike_lakehouse/bronze/source_systems/{table}.csv")
        )

        df = df.withColumn("_load_timestamp",current_timestamp()).withColumn("_source_system",lit(source_system))

        df.write.mode("overwrite").saveAsTable(f"bike_lakehouse.bronze.{source_system}_{table}")



for source_system,details in config["source_systems"].items():

    load_to_bronze(details["tables"],source_system)




In [0]:
from pyspark.sql.functions import current_timestamp, lit
import yaml


with open("/Workspace/Users/kahledtrojan@gmail.com/databricks_lakehouse_pipeline_de/bike_lakehouse/config.yaml","r") as f:
    config = yaml.safe_load(f)


def load_to_bronze(tables,source_system):
    for table in tables:
        df = (
            spark.read.option("header","true")
            .option("inferSchema","true")
            .csv(f"/Volumes/bike_lakehouse/bronze/source_systems/{table}.csv")
        )

        df = df.withColumn("_load_timestamp",current_timestamp()).withColumn("_source_system",lit(source_system))

        df.write.mode("overwrite").saveAsTable(f"bike_lakehouse.bronze.{source_system}_{table}")



for source_system,details in config["source_systems"].items():

    load_to_bronze(details["tables"],source_system)

In [0]:
%sql
DROP TABLE IF EXISTS bike_lakehouse.bronze.crm_cust_info;
DROP TABLE IF EXISTS  bike_lakehouse.bronze.crm_prd_info;
DROP TABLE IF EXISTS  bike_lakehouse.bronze.crm_sales_details;
DROP TABLE IF EXISTS  bike_lakehouse.bronze.erp_CUST_AZ12;
DROP TABLE IF EXISTS bike_lakehouse.bronze.erp_LOC_A101;
DROP TABLE IF EXISTS bike_lakehouse.bronze.erp_PX_CAT_G1V2;


## extract from database

In [0]:
from pyspark.sql.functions import current_timestamp, lit
import yaml
from dotenv import load_dotenv
import os


In [0]:
base_path = "/Workspace/Users/kahledtrojan@gmail.com/databricks_lakehouse_pipeline_de/bike_lakehouse/config"
environment = "dev"

In [0]:
load_dotenv(
    "/Workspace/Users/kahledtrojan@gmail.com/databricks_lakehouse_pipeline_de/bike_lakehouse/config/dev/secrets/.env"
)


In [0]:
with open(f"{base_path}/{environment}/config.yaml","r") as f:
    config = yaml.safe_load(f)


with open(f"{base_path}/{environment}/database.yaml","r") as f:
    database = yaml.safe_load(f)

### secrets in code

In [0]:
# jdbc_url = (
#     "jdbc:postgresql://ep-rapid-darkness-a5qejnez-pooler.us-east-2.aws.neon.tech:5432/neondb"
# )




# properties = {
#     "user": "neondb_owner",
#     "password": "npg_MU9f0FxKVeDo",
#     "driver": "org.postgresql.Driver"
# }
# def load_to_bronze(tables,source_system):

#     for table,table_mode in tables.items():
#         mode = table_mode.get("load_mode", "append")
        
#         df = (
#             spark.read
#             .jdbc(
#                 url=jdbc_url,
#                 table=F"{source_system}_{table}",
#                 properties=properties
#             )
#         )

#         df = df.withColumn("_load_timestamp",current_timestamp()).withColumn("_source_system",lit(source_system))

#         df.write.mode(mode).saveAsTable(f"bike_lakehouse.bronze.{source_system}_{table}")


# for source_system,details in config["source_systems"].items():

#     load_to_bronze(details["tables"],source_system)


### ### secrets in .env

In [0]:
jdbc_url = (
    f"jdbc:postgresql://{os.getenv("POSTGRES_HOST")}:{os.getenv("POSTGRES_PORT")}/{os.getenv("POSTGRES_DATABASE")}"
)



properties = {
    "user": os.getenv("POSTGRES_USERNAME"),
    "password": os.getenv("POSTGRES_PASSWORD"),
    "driver": "org.postgresql.Driver"
}
def load_to_bronze(tables,source_system):

    for table,table_mode in tables.items():
        mode = table_mode.get("load_mode","append")
        
        df = (
            spark.read
            .jdbc(
                url=jdbc_url,
                table=F"{source_system}_{table}",
                properties=properties
            )
        )

        df = df.withColumn("_load_timestamp",current_timestamp()).withColumn("_source_system",lit(source_system))

        df.write.mode(mode).saveAsTable(f"bike_lakehouse.bronze.{source_system}_{table}")


for source_system,details in config["source_systems"].items():

    load_to_bronze(details["tables"],source_system)


### secrets in yaml

In [0]:
jdbc_url = (
    f"jdbc:postgresql://{database["postgresqL"]["host"]}:5432/{database["postgresqL"]["database"]}"
)



properties = {
    "user": database["postgresqL"]["username"],
    "password": database["postgresqL"]["password"],
    "driver": "org.postgresql.Driver"
}
def load_to_bronze(tables,source_system):

    for table,table_mode in tables.items():
        mode = table_mode.get("load_mode","append")
        
        df = (
            spark.read
            .jdbc(
                url=jdbc_url,
                table=F"{source_system}_{table}",
                properties=properties
            )
        )

        df = df.withColumn("_load_timestamp",current_timestamp()).withColumn("_source_system",lit(source_system))

        df.write.mode(mode).saveAsTable(f"bike_lakehouse.bronze.{source_system}_{table}")


for source_system,details in config["source_systems"].items():

    load_to_bronze(details["tables"],source_system)


In [0]:


# jdbc_url =(
#      "jdbc:postgresql://" + dbutils.secrets.get(scope="bike_lakehouse.bronze.postgres_scope",key="host") + ":5432/neondb"
# )

# properties = {
#       "users":dbutils.secrets.get(scope = "bike_lakehouse.bronze.postgres_scope",key="username"),
#       "password":dbutils.secrets.get(scope = "bike_lakehouse.bronze.postgres_scope",key="password"),
#       "driver": "org.postgresql.Driver"
# }


# def load_to_bronze(tables,source_system):

#     for table,table_mode in tables.items():
#         mode = table_mode["load_mode"]
        
#         df = (
#             spark.read
#             .jdbc(
#                 url=jdbc_url,
#                 table=F"{source_system}_{table}",
#                 properties=properties
#             )
#         )

#         df = df.withColumn("_load_timestamp",current_timestamp()).withColumn("_source_system",lit(source_system))

#         df.write.mode(mode).saveAsTable(f"bike_lakehouse.bronze.{source_system}_{table}")


# for source_system,details in config["source_systems"].items():

#     load_to_bronze(details["tables"],source_system)
